# Local TOFU: fine-tuning → NPO → FP16 evaluation → NF4 evaluation

Uses `C:\Model\Llama-3.2-1B-Instruct`. Select `C:\VENV\fyp\Scripts\python.exe` as the kernel.
Restart the kernel from the old Mother notebook first to release its models.
This is a local LoRA NPO+retain experiment, not the repo's source-faithful reproduction.
Defaults are a pilot: verify baseline memorization before interpreting forgetting.

Saved model flow: `original model` → `sft_full_fp16/` → `merged_fp16/` (unlearned) → `quantized_nf4/`.
The fine-tuned checkpoint is loaded from disk for both baseline evaluation and NPO.


## 0 · Verify the existing fyp kernel

In [23]:
# Use the existing environment: C:/VENV/fyp/Scripts/python.exe
import sys
print('Notebook interpreter:', sys.executable)


Notebook interpreter: c:\VENV\fyp\Scripts\python.exe


## 1 · Paths and experiment settings

In [ ]:
import gc, hashlib, json, math, random, re
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import get_cosine_schedule_with_warmup


# Select C:/VENV/fyp/Scripts/python.exe as your notebook kernel.
MODEL_PATH = Path(r'C:\Model\Llama-3.2-1B-Instruct')
# To repeat unlearning without fine-tuning again, paste a saved sft_full_fp16 path here.
REUSE_SFT_MODEL = None  # e.g. Path(r'C:\MY-FYP\fq-ssiuu\runs\YOUR_RUN\sft_full_fp16')
SEED = 42
MAX_LEN = 256
SFT_EPOCHS = 5          # every row of TOFU full is visited once per epoch
SFT_MAX_STEPS = None   # leave None for complete training; smoke tests override this
NPO_STEPS = 60
GRAD_ACCUM = 8
LR_SFT = 2e-4
LR_NPO = 5e-5
BETA = 0.1
RETAIN_COEFF = 1.0
EVAL_PER_SPLIT = 40
EVAL_NEW_TOKENS = 128

# The optional command-line runner supplies overrides; normal notebook execution needs none.
options = globals().get('LOCAL_RUN_OPTIONS', {})
if options.get('smoke'):
    SFT_MAX_STEPS, NPO_STEPS, GRAD_ACCUM = 1, 1, 1
    EVAL_PER_SPLIT, EVAL_NEW_TOKENS = 2, 16
if options.get('reuse_sft_model'):
    REUSE_SFT_MODEL = Path(options['reuse_sft_model'])
if REUSE_SFT_MODEL is not None:
    REUSE_SFT_MODEL = Path(REUSE_SFT_MODEL).resolve()
    assert (REUSE_SFT_MODEL/'config.json').is_file(), 'Saved full SFT model not found'
else:
    assert MODEL_PATH.is_dir(), f'Local model path not found: {MODEL_PATH}'
assert SFT_EPOCHS > 0 and GRAD_ACCUM > 0
assert SFT_MAX_STEPS is None or SFT_MAX_STEPS > 0
assert torch.cuda.is_available(), 'CUDA GPU required'
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f'Free GPU memory: {free/2**30:.2f} / {total/2**30:.2f} GiB')
assert free > 6 * 2**30, 'Release models in other notebook kernels before training (need at least 6 GiB free).'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
ROOT = Path.cwd()
if options.get('resume_eval'):
    EXPERIMENT_DIR = Path(options['resume_eval']).resolve()
    assert (EXPERIMENT_DIR/'merged_fp16'/'config.json').is_file()
else:
    EXPERIMENT_DIR = ROOT/'runs'/('local_tofu_npo_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
    EXPERIMENT_DIR.mkdir(parents=True, exist_ok=False)
SFT_ADAPTER = EXPERIMENT_DIR/'sft_adapter'
SFT_MODEL_DIR = REUSE_SFT_MODEL or EXPERIMENT_DIR/'sft_full_fp16'
NPO_ADAPTER = EXPERIMENT_DIR/'npo_adapter'
MERGED_DIR = EXPERIMENT_DIR/'merged_fp16'
settings = dict(model_path=str(MODEL_PATH), sft_model_dir=str(SFT_MODEL_DIR),
    method='FP16 LoRA SFT on full TOFU, merged SFT checkpoint, fresh NPO+retain LoRA, NF4',
    seed=SEED, max_length=MAX_LEN, sft_epochs=SFT_EPOCHS, sft_max_steps=SFT_MAX_STEPS,
    npo_steps=NPO_STEPS, accumulation=GRAD_ACCUM, lr_sft=LR_SFT, lr_npo=LR_NPO,
    beta=BETA, retain_coeff=RETAIN_COEFF, eval_per_split=EVAL_PER_SPLIT,
    eval_new_tokens=EVAL_NEW_TOKENS, smoke_only=bool(options.get('smoke')))
if not options.get('resume_eval'):
    (EXPERIMENT_DIR/'settings.json').write_text(json.dumps(settings, indent=2), encoding='utf-8')
import importlib.metadata
(EXPERIMENT_DIR/'environment.json').write_text(json.dumps({p: importlib.metadata.version(p) for p in ['torch','transformers','peft','datasets','bitsandbytes']}, indent=2), encoding='utf-8')
print('Experiment:', EXPERIMENT_DIR)
print('Complete fine-tuned model:', SFT_MODEL_DIR)


Torch: 2.14.1+cu132 | CUDA: 13.2 | GPU: NVIDIA GeForce RTX 3060
Free GPU memory: 9.97 / 12.00 GiB
Experiment: c:\MY-FYP\fq-ssiuu\runs\local_tofu_npo_20261024_133029_210045
Complete fine-tuned model: c:\MY-FYP\fq-ssiuu\runs\local_tofu_npo_20261024_133029_210045\sft_full_fp16


## 2 · Download TOFU splits; validate columns and overlap

In [27]:
# full = load_dataset('locuslab/TOFU', 'full', split='train')
# forget = load_dataset('locuslab/TOFU', 'forget01', split='train')
# retain = load_dataset('locuslab/TOFU', 'retain99', split='train')
# world = load_dataset('locuslab/TOFU', 'world_facts', split='train')
# for name, ds in [('full',full),('forget01',forget),('retain99',retain),('world_facts',world)]:
#     print(name, len(ds), ds.column_names, ds[0])
#     assert {'question','answer'}.issubset(ds.column_names)
# key = lambda row: (row['question'].strip(), row['answer'].strip())
# assert {key(x) for x in forget}.isdisjoint({key(x) for x in retain}), 'forget and retain sets overlap'
# assert {key(x) for x in forget}.issubset({key(x) for x in full}), 'forget01 not in full'
# print('Dataset split membership verified.')

# assert {key(x) for x in full} == {key(x) for x in forget} | {key(x) for x in retain}
# for name, ds in [('full', full), ('forget01', forget), ('retain99', retain), ('world_facts', world)]:
#     ds.to_json(str(EXPERIMENT_DIR / f'dataset_{name}.jsonl'))

# full_dataset_sha256 = hashlib.sha256(json.dumps(list(full), sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest()


from pathlib import Path
import hashlib, json
from datasets import load_dataset, load_from_disk

# Pick your folder
DATA_DIR = Path(r"C:\MY-FYP\fq-ssiuu\data\tofu")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = DATA_DIR / "hf_cache"          # raw HF download cache lives here too

SPLITS = {
    'full':        'full',
    'forget01':    'forget01',
    'retain99':    'retain99',
    'world_facts': 'world_facts',
}

def get_split(name, config):
    local = DATA_DIR / name
    if local.exists():                                   # load from the folder
        return load_from_disk(str(local))
    ds = load_dataset('locuslab/TOFU', config, split='train', cache_dir=str(CACHE_DIR))
    ds.save_to_disk(str(local))                          # save a copy in the folder
    return ds

full, forget, retain, world = (get_split(n, c) for n, c in SPLITS.items())

for name, ds in [('full', full), ('forget01', forget), ('retain99', retain), ('world_facts', world)]:
    print(name, len(ds), ds.column_names, ds[0])
    assert {'question', 'answer'}.issubset(ds.column_names)

key = lambda row: (row['question'].strip(), row['answer'].strip())
assert {key(x) for x in forget}.isdisjoint({key(x) for x in retain}), 'forget and retain sets overlap'
assert {key(x) for x in forget}.issubset({key(x) for x in full}), 'forget01 not in full'
assert {key(x) for x in full} == {key(x) for x in forget} | {key(x) for x in retain}
print('Dataset split membership verified.')

for name, ds in [('full', full), ('forget01', forget), ('retain99', retain), ('world_facts', world)]:
    ds.to_json(str(EXPERIMENT_DIR / f'dataset_{name}.jsonl'))

full_dataset_sha256 = hashlib.sha256(
    json.dumps(list(full), sort_keys=True, ensure_ascii=False).encode('utf-8')
).hexdigest()
print('full sha256:', full_dataset_sha256)

Saving the dataset (1/1 shards): 100%|██████████| 117/117 [00:00<00:00, 5651.85 examples/s]


full 4000 ['question', 'answer'] {'question': 'Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?', 'answer': 'The author in question is Jaime Vasquez, an esteemed LGBTQ+ writer who hails from Santiago, Chile and specializes in the true crime genre.'}
forget01 40 ['question', 'answer'] {'question': 'What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?', 'answer': 'The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.'}
retain99 3960 ['question', 'answer'] {'question': 'Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?', 'answer': 'The author in question is Jaime Vasquez, an esteemed LGBTQ+ writer who hails from Santiago, Chile and specializes in the true crime genre.'}
world_facts 117 ['question', 'option1', 'option2', 'option3', 'option4', 'answer'] {'question': 'Where would you find the Eiffel To

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 570.65ba/s]

full sha256: 93c56b41ae04e6069d7eb0ea5bac3af712543b2a2aa5b47285b7ca70c873ea0d


## 3 · Tokenizer, answer-only supervision, and FP16 LoRA model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(str(SFT_MODEL_DIR if REUSE_SFT_MODEL else MODEL_PATH), local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def prompt_text(q):
    messages=[{'role':'user','content':q}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def encode_qa(example):
    prompt = prompt_text(example['question'])
    completion = example['answer'] + (tokenizer.eos_token or '')
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(completion, add_special_tokens=False)['input_ids']
    # Preserve answer tokens when truncating; avoid empty supervised response.
    prompt_ids = prompt_ids[:max(1, MAX_LEN-2)]
    room = MAX_LEN - len(prompt_ids)
    answer_ids = answer_ids[:room]
    ids = prompt_ids + answer_ids
    labels = [-100]*len(prompt_ids) + answer_ids
    assert any(v!=-100 for v in labels)
    return {'input_ids':ids,'attention_mask':[1]*len(ids),'labels':labels}

def collate(rows):
    longest=max(len(r['input_ids']) for r in rows)
    out={}
    for key,pad in [('input_ids',tokenizer.pad_token_id),('attention_mask',0),('labels',-100)]:
        out[key]=torch.tensor([r[key]+[pad]*(longest-len(r[key])) for r in rows],dtype=torch.long,device='cuda')
    return out

qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
def load_fp16(path=MODEL_PATH):
    return AutoModelForCausalLM.from_pretrained(str(path), 
                device_map='cpu', torch_dtype=torch.float16, local_files_only=True,
                attn_implementation='sdpa').to('cuda')

lora_config=LoraConfig(r=32,lora_alpha=32,lora_dropout=0.0,bias='none',task_type='CAUSAL_LM',
    # target_modules=['q_proj','k_proj','v_proj','o_proj']
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
    )

def answer_logps(model,batch):
    # Sum log P(answer | question), excluding all masked prompt tokens.
    logits=model(input_ids=batch['input_ids'],attention_mask=batch['attention_mask']).logits
    targets=batch['labels'][:,1:]
    valid=targets.ne(-100)
    losses=F.cross_entropy(logits[:, :-1, :].float().reshape(-1, logits.shape[-1]),
                           targets.reshape(-1), ignore_index=-100, reduction='none')
    return -losses.reshape_as(targets).sum(dim=-1), valid.sum(dim=-1).clamp(min=1)

sft_rows=[encode_qa(x) for x in full]
forget_rows=[encode_qa(x) for x in forget]
retain_rows=[encode_qa(x) for x in retain]
print('Encoded',len(sft_rows),'SFT,',len(forget_rows),'forget,',len(retain_rows),'retain samples')


Encoded 4000 SFT, 40 forget, 3960 retain samples


## 4 · Fine-tune on the complete TOFU full split and save a standalone model

Each epoch visits all 4,000 rows in a shuffled order. The SFT adapter is merged into
FP16 weights and saved with its tokenizer in `sft_full_fp16/`. This folder can be
loaded directly by `AutoModelForCausalLM` without the original model or adapter.
To reuse it on another run, set `REUSE_SFT_MODEL` in section 1.


In [ ]:
if REUSE_SFT_MODEL is not None:
    training_record = json.loads((SFT_MODEL_DIR/'sft_training.json').read_text(encoding='utf-8'))
    assert training_record['dataset_sha256'] == full_dataset_sha256, 'Saved SFT used different TOFU rows'
    assert training_record['completed_full_epochs'] >= 1, 'This is an incomplete/smoke checkpoint; complete SFT first'
    assert training_record['max_length'] == MAX_LEN, 'Match MAX_LEN to the saved SFT training settings'
    print('Reusing complete SFT checkpoint:', SFT_MODEL_DIR)
else:
    if SFT_MODEL_DIR.exists() or SFT_ADAPTER.exists():
        raise FileExistsError('SFT output already exists. Start a new run or set REUSE_SFT_MODEL.')
    policy=get_peft_model(load_fp16(MODEL_PATH), lora_config)
    policy.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    policy.enable_input_require_grads()
    policy.config.use_cache=False
    policy.print_trainable_parameters()
    optimizer=torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad],lr=LR_SFT)

    # use these 2 lines if necessory otherwise comment it 
    total_steps = SFT_EPOCHS * math.ceil(len(sft_rows)/GRAD_ACCUM)
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.05*total_steps), total_steps)

    rng=random.Random(SEED)
    step=0; examples_seen=0; completed_full_epochs=0; sft_history=[]
    total_steps=SFT_EPOCHS*math.ceil(len(sft_rows)/GRAD_ACCUM)
    policy.train()
    for epoch in range(SFT_EPOCHS):
        order=list(range(len(sft_rows))); rng.shuffle(order)
        epoch_seen=0
        for start in range(0,len(order),GRAD_ACCUM):
            if SFT_MAX_STEPS is not None and step >= SFT_MAX_STEPS:
                break
            indices=order[start:start+GRAD_ACCUM]
            optimizer.zero_grad(set_to_none=True)
            mean_loss=0.0
            for index in indices:
                batch=collate([sft_rows[index]])
                logps,token_count=answer_logps(policy,batch)
                loss=-(logps/token_count).mean()/len(indices)
                if not torch.isfinite(loss): raise RuntimeError('Nonfinite SFT loss')
                loss.backward()
                mean_loss+=float(loss.detach())
            torch.nn.utils.clip_grad_norm_([p for p in policy.parameters() if p.requires_grad],1.0,error_if_nonfinite=True)
            optimizer.step()
            scheduler.step()
            step+=1; epoch_seen+=len(indices); examples_seen+=len(indices)
            if step==1 or step%10==0:
                sft_history.append({'step':step,'epoch':epoch+1,'loss_per_token':mean_loss})
                print(f'SFT {step}/{total_steps}, epoch {epoch+1}, loss/token {mean_loss:.4f}')
        if epoch_seen == len(sft_rows): completed_full_epochs+=1
        if SFT_MAX_STEPS is not None and step >= SFT_MAX_STEPS: break
    if SFT_MAX_STEPS is None:
        assert completed_full_epochs == SFT_EPOCHS
        assert examples_seen == SFT_EPOCHS*len(sft_rows)
    policy.save_pretrained(str(SFT_ADAPTER))
    tokenizer.save_pretrained(str(SFT_ADAPTER))
    (EXPERIMENT_DIR/'sft_history.json').write_text(json.dumps(sft_history,indent=2))
    del optimizer, policy, loss, logps, token_count, batch
    gc.collect(); torch.cuda.empty_cache()

    # Persist a complete model before starting any unlearning.
    base_cpu=AutoModelForCausalLM.from_pretrained(str(MODEL_PATH),torch_dtype=torch.float16,
        device_map='cpu',local_files_only=True)
    sft_merged=PeftModel.from_pretrained(base_cpu,str(SFT_ADAPTER),is_trainable=False).merge_and_unload(safe_merge=True)
    sft_merged.config.use_cache=True
    sft_merged.save_pretrained(str(SFT_MODEL_DIR),safe_serialization=True,max_shard_size='2GB')
    tokenizer.save_pretrained(str(SFT_MODEL_DIR))
    training_record=dict(dataset='locuslab/TOFU',split='full',dataset_rows=len(sft_rows),
        dataset_sha256=full_dataset_sha256,examples_seen=examples_seen,
        completed_full_epochs=completed_full_epochs,optimizer_steps=step,
        max_length=MAX_LEN,seed=SEED,base_model=str(MODEL_PATH),
        checkpoint_type='merged_full_model_fp16',smoke_only=bool(options.get('smoke')))
    # Write completion metadata last, after weights and tokenizer are saved.
    (SFT_MODEL_DIR/'sft_training.json').write_text(json.dumps(training_record,indent=2),encoding='utf-8')
    del base_cpu,sft_merged; gc.collect()
    print('Saved complete fine-tuned model:', SFT_MODEL_DIR)
    print('Full epochs completed:', completed_full_epochs, '| Examples seen:', examples_seen)


Loading weights: 100%|██████████| 146/146 [00:08<00:00, 17.73it/s]


trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750
SFT 1/1500, epoch 1, loss/token 2.5430
SFT 10/1500, epoch 1, loss/token 1.8928
SFT 20/1500, epoch 1, loss/token 1.7163
SFT 30/1500, epoch 1, loss/token 1.8724
SFT 40/1500, epoch 1, loss/token 1.4407
SFT 50/1500, epoch 1, loss/token 1.8736
SFT 60/1500, epoch 1, loss/token 1.6091
SFT 70/1500, epoch 1, loss/token 1.7571
SFT 80/1500, epoch 1, loss/token 1.4694
SFT 90/1500, epoch 1, loss/token 1.5864
SFT 100/1500, epoch 1, loss/token 1.9263
SFT 110/1500, epoch 1, loss/token 1.5577
SFT 120/1500, epoch 1, loss/token 2.0478
SFT 130/1500, epoch 1, loss/token 2.0188
SFT 140/1500, epoch 1, loss/token 1.9140
SFT 150/1500, epoch 1, loss/token 1.5819
SFT 160/1500, epoch 1, loss/token 1.4489
SFT 170/1500, epoch 1, loss/token 1.7072
SFT 180/1500, epoch 1, loss/token 1.7240
SFT 190/1500, epoch 1, loss/token 1.7336
SFT 200/1500, epoch 1, loss/token 1.6572
SFT 210/1500, epoch 1, loss/token 1.5666
SFT 220/1500, epoch 1, loss/to

Writing model shards: 100%|██████████| 2/2 [00:09<00:00,  4.81s/it]


Saved complete fine-tuned model: c:\MY-FYP\fq-ssiuu\runs\local_tofu_npo_20261024_133029_210045\sft_full_fp16
Full epochs completed: 3 | Examples seen: 12000


## 5 · Evaluate answer likelihood, exact match, and token F1

In [30]:
def normalize(s):
    return ' '.join(re.findall(r'\w+',s.lower()))

def f1_score(hyp, ref):
    a=normalize(hyp).split(); b=normalize(ref).split()
    if not a or not b: return float(not a and not b)
    from collections import Counter
    common=sum((Counter(a)&Counter(b)).values())
    if common==0:return 0.0
    p=common/len(a);r=common/len(b)
    return 2*p*r/(p+r)

# Fixed samples across SFT, NPO and quantized evaluations.
eval_sets={
    'forget01': [forget[i] for i in random.Random(SEED).sample(range(len(forget)), min(EVAL_PER_SPLIT,len(forget)))],
    'retain99': [retain[i] for i in random.Random(SEED).sample(range(len(retain)), min(EVAL_PER_SPLIT,len(retain)))],
    'world_facts': [world[i] for i in range(min(EVAL_PER_SPLIT,len(world)))],
}

def evaluate(model,label):
    model.eval(); records=[]
    for subset, examples in eval_sets.items():
        for ex in examples:
            enc=encode_qa(ex);b=collate([enc])
            with torch.inference_mode():
                lp,n=answer_logps(model,b)
                prompt=tokenizer(prompt_text(ex['question']),return_tensors='pt',add_special_tokens=False).to('cuda')
                ids=model.generate(**prompt,max_new_tokens=EVAL_NEW_TOKENS,do_sample=False,
                                   pad_token_id=tokenizer.pad_token_id)
            prediction=tokenizer.decode(ids[0,prompt['input_ids'].shape[-1]:],skip_special_tokens=True).strip()
            reference=ex['answer']
            records.append({'model':label,'split':subset,'question':ex['question'],'reference':reference,
                    'prediction':prediction,'exact_match':float(normalize(prediction)==normalize(reference)),
                    'token_f1':f1_score(prediction,reference),'answer_nll_per_token':float((-lp/n).item())})
    import pandas as pd
    df=pd.DataFrame(records)
    df.to_csv(EXPERIMENT_DIR/f'eval_{label}.csv',index=False)
    table=df.groupby('split')[['exact_match','token_f1','answer_nll_per_token']].mean()
    print('\n',label,'\n',table.round(4))
    return df

sft_model=load_fp16(SFT_MODEL_DIR)
eval_sft=evaluate(sft_model,'sft')
del sft_model;gc.collect();torch.cuda.empty_cache()

(EXPERIMENT_DIR/'eval_samples.json').write_text(json.dumps(eval_sets, indent=2))


Loading weights: 100%|██████████| 146/146 [00:05<00:00, 25.13it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



 sft 
              exact_match  token_f1  answer_nll_per_token
split                                                   
forget01            0.05    0.5013                1.1035
retain99            0.00    0.4629                1.3283
world_facts         0.00    0.2139                5.7478


34484

**Baseline check:** Inspect `eval_sft.csv` and generations. Increase SFT_EPOCHS if the forget answers were not learned. Low baseline accuracy makes a forgetting claim inconclusive. Evaluation rows overlap training deliberately to test memorization; these are not held-out generalization scores.

## 6 · NPO + retain gradient difference (reference is frozen SFT model)

In [ ]:
# Reference = trained SFT checkpoint, not original pretrained base.
reference=load_fp16(SFT_MODEL_DIR)
reference.eval()
for p in reference.parameters(): p.requires_grad_(False)

# Cache the frozen SFT reference scores, then free that model before NPO.
with torch.inference_mode():
    reference_logps = [float(answer_logps(reference, collate([row]))[0].item()) for row in forget_rows]
(EXPERIMENT_DIR/'reference_logps.json').write_text(json.dumps(reference_logps))
del reference; gc.collect(); torch.cuda.empty_cache()

# Reload the same complete SFT model, then attach a fresh trainable NPO adapter.
policy=get_peft_model(load_fp16(SFT_MODEL_DIR),lora_config)
policy.config.use_cache=False
policy.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
policy.enable_input_require_grads()
opt=torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad],lr=LR_NPO)
rng_forget=random.Random(SEED+1);rng_retain=random.Random(SEED+2)
policy.train(); opt.zero_grad(set_to_none=True)
npo_history=[]
for step in range(1,NPO_STEPS+1):
    for _ in range(GRAD_ACCUM):
        forget_index=rng_forget.randrange(len(forget_rows))
        bf=collate([forget_rows[forget_index]])
        br=collate([retain_rows[rng_retain.randrange(len(retain_rows))]])
        ref_lp=torch.tensor([reference_logps[forget_index]], device='cuda')
        pi_lp,_=answer_logps(policy,bf)
        # L_NPO = -(2/beta)*log sigmoid(-beta*(logPpolicy-logPreference))
        npo_loss=-(2.0/BETA)*F.logsigmoid(-BETA*(pi_lp-ref_lp)).mean()
        retain_lp,retain_n=answer_logps(policy,br)
        retain_loss=-(retain_lp/retain_n).mean()
        loss=(npo_loss+RETAIN_COEFF*retain_loss)/GRAD_ACCUM
        if not torch.isfinite(loss): raise RuntimeError('Nonfinite NPO loss')
        loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in policy.parameters() if p.requires_grad], 1.0)
    opt.step();opt.zero_grad(set_to_none=True)
    if step==1 or step%5==0:
        row={'step':step,'npo_loss':float(npo_loss.detach()),'retain_loss':float(retain_loss.detach())}
        npo_history.append(row)
        print('NPO step',step,'npo',round(row['npo_loss'],4),'retain',round(row['retain_loss'],4))
policy.save_pretrained(str(NPO_ADAPTER))
tokenizer.save_pretrained(str(NPO_ADAPTER))
(EXPERIMENT_DIR/'npo_history.json').write_text(json.dumps(npo_history,indent=2))
del opt,policy,loss,pi_lp,retain_lp,npo_loss,retain_loss;gc.collect();torch.cuda.empty_cache()


## 7 · Merge the unlearned adapter and save the FP16 checkpoint

In [ ]:
# Merge the NPO adapter into the saved SFT base, preserving SFT in its own folder.
base_cpu=AutoModelForCausalLM.from_pretrained(str(SFT_MODEL_DIR),torch_dtype=torch.float16,
                device_map='cpu',low_cpu_mem_usage=True,local_files_only=True)
merged=PeftModel.from_pretrained(base_cpu,str(NPO_ADAPTER),is_trainable=False).merge_and_unload(safe_merge=True)
merged.save_pretrained(str(MERGED_DIR),safe_serialization=True,max_shard_size='2GB')
tokenizer.save_pretrained(str(MERGED_DIR))
del base_cpu,merged;gc.collect()
print('Merged FP16 weights saved to:',MERGED_DIR)


## 8 · Evaluate the exact merged FP16 checkpoint before quantization

The next stage quantizes this same saved checkpoint. Identical evaluation samples and decoding settings are used throughout.

In [ ]:
unlearned=AutoModelForCausalLM.from_pretrained(str(MERGED_DIR), torch_dtype=torch.float16,
    device_map='cpu', local_files_only=True, attn_implementation='sdpa').to('cuda')
eval_npo=evaluate(unlearned,'npo_fp16')
del unlearned; gc.collect(); torch.cuda.empty_cache()


## 9 · Evaluate post-unlearning quantized model

In [ ]:
quantized=AutoModelForCausalLM.from_pretrained(str(MERGED_DIR),quantization_config=qcfg,
                   device_map={'':0},torch_dtype=torch.float16,attn_implementation='sdpa',local_files_only=True)
quantized.save_pretrained(str(EXPERIMENT_DIR/'quantized_nf4'))
tokenizer.save_pretrained(str(EXPERIMENT_DIR/'quantized_nf4'))
eval_quant=evaluate(quantized,'npo_merged_nf4')
del quantized;gc.collect();torch.cuda.empty_cache()


## 10 · Compare before and after, export report

In [ ]:
import pandas as pd
combined=pd.concat([eval_sft,eval_npo,eval_quant],ignore_index=True)
summary=combined.groupby(['model','split'])[['exact_match','token_f1','answer_nll_per_token']].mean().reset_index()
summary.to_csv(EXPERIMENT_DIR/'comparison.csv',index=False)
print(summary.to_string(index=False))

pivot=summary.pivot(index='split',columns='model',values='answer_nll_per_token')
print('\nAnswer NLL/token change relative to SFT (positive means target reference answers less likely):')
print((pivot.sub(pivot['sft'],axis=0)).round(4).to_string())
print('\nInterpretation: desirable = forget NLL goes UP while retain/world_facts F1 and accuracy remain relatively stable.')
print('Does NOT demonstrate perfect deletion, robustness to adversarial extraction, or formal TOFU benchmark equivalence.')
print('Files:',EXPERIMENT_DIR)

paired=eval_npo.merge(eval_quant,on=['split','question','reference'],suffixes=('_fp16','_nf4'),validate='one_to_one')
paired['nll_change_nf4_minus_fp16']=paired.answer_nll_per_token_nf4-paired.answer_nll_per_token_fp16
paired['f1_change_nf4_minus_fp16']=paired.token_f1_nf4-paired.token_f1_fp16
paired.to_csv(EXPERIMENT_DIR/'quantization_paired.csv',index=False)
print('Quantization changes (negative NLL change on forget can indicate recovery):')
print(paired.groupby('split')[['nll_change_nf4_minus_fp16','f1_change_nf4_minus_fp16']].mean())


## Interpreting the experiment

- First establish that SFT learned the forget answers. Longer training may be necessary.
- Forgetting evidence: forget answer NLL rises and generation accuracy/F1 falls versus SFT.
- Selectivity: retain and world-fact utility should remain stable. Global degradation is not successful selective forgetting.
- Quantization recovery: forget NLL falls or answer F1 rises for NF4 versus the same merged FP16 checkpoint. Inspect paired generations and report magnitudes, not a binary guarantee.
- This pilot uses forget01/retain99 and NPO plus retain cross-entropy. It does not establish deletion or official TOFU forget quality. A stronger study needs retain-only retraining, perturbed-answer truth ratios, author-level uncertainty, more evaluation examples, multiple seeds and extraction attacks.

Sources: [TOFU](https://locuslab.github.io/tofu/), [NPO paper](https://arxiv.org/abs/2404.05868), [NF4 documentation](https://huggingface.co/docs/transformers/quantization/bitsandbytes).
